In [1]:
import pandas as pd

from collections import defaultdict

In [2]:
file_path = "/content/Market_Basket_Optimisation.csv"

df = pd.read_csv(

    file_path,

    header=None

)

print("Dataset Shape:", df.shape)

display(df.head())

Dataset Shape: (7501, 20)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
transactions = []

for i in range(len(df)):

    transaction = []

    for item in df.iloc[i]:

        if pd.notna(item):

            transaction.append(str(item).strip())

    transactions.append(transaction)

print("Total Transactions:", len(transactions))

print("\nFirst 5 Transactions:")

for transaction in transactions[:5]:

    print(transaction)

Total Transactions: 7501

First 5 Transactions:
['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil']
['burgers', 'meatballs', 'eggs']
['chutney']
['turkey', 'avocado']
['mineral water', 'milk', 'energy bar', 'whole wheat rice', 'green tea']


In [4]:
min_support = 0.01

total_transactions = len(transactions)

min_support_count = int(

    min_support * total_transactions

)

print("Minimum Support:", min_support)

print("Minimum Support Count:", min_support_count)

Minimum Support: 0.01
Minimum Support Count: 75


In [5]:
item_counts = defaultdict(int)

for transaction in transactions:

    for item in transaction:

        item_counts[item] += 1

# Remove items below minimum support

frequent_items = {

    item: count

    for item, count in item_counts.items()

    if count >= min_support_count

}

print(

    "Number of Frequent Items:",

    len(frequent_items)

)

Number of Frequent Items: 75


In [6]:
sorted_transactions = []

for transaction in transactions:

    filtered_transaction = [

        item

        for item in transaction

        if item in frequent_items

    ]

    filtered_transaction.sort(

        key=lambda x: frequent_items[x],

        reverse=True

    )

    if filtered_transaction:

        sorted_transactions.append(

            filtered_transaction

        )

print("Sorted Transactions:")

for transaction in sorted_transactions[:5]:

    print(transaction)

Sorted Transactions:
['mineral water', 'green tea', 'low fat yogurt', 'shrimp', 'olive oil', 'frozen smoothie', 'honey', 'salmon', 'avocado', 'cottage cheese', 'tomato juice', 'energy drink', 'vegetables mix', 'almonds', 'yams']
['eggs', 'burgers', 'meatballs']
['turkey', 'avocado']
['mineral water', 'green tea', 'milk', 'whole wheat rice', 'energy bar']
['low fat yogurt']


In [7]:
class FPNode:

    def __init__(

        self,

        item,

        count=1,

        parent=None

    ):

        self.item = item

        self.count = count

        self.parent = parent

        self.children = {}

    def increment(self, count=1):

        self.count += count

In [8]:
root = FPNode("ROOT")

header_table = {}

for transaction in sorted_transactions:

    current_node = root

    for item in transaction:

        if item in current_node.children:

            current_node.children[item].increment()

        else:

            new_node = FPNode(

                item,

                1,

                current_node

            )

            current_node.children[item] = new_node

        current_node = current_node.children[item]

        if item not in header_table:

            header_table[item] = []

        header_table[item].append(

            current_node

        )

print("FP-Tree Created Successfully")

FP-Tree Created Successfully


In [9]:
frequent_itemsets = {}

for item in frequent_items:

    total_count = 0

    for node in header_table[item]:

        total_count += node.count

    frequent_itemsets[

        frozenset([item])

    ] = total_count

In [10]:
from itertools import combinations

all_items = list(frequent_items.keys())

for size in range(2, 4):

    for combination in combinations(

        all_items,

        size

    ):

        itemset = frozenset(combination)

        count = 0

        for transaction in transactions:

            if itemset.issubset(

                set(transaction)

            ):

                count += 1

        if count >= min_support_count:

            frequent_itemsets[

                itemset

            ] = count

In [11]:


results = []

for itemset, count in frequent_itemsets.items():

    support = (
        count /
        total_transactions
    )

    results.append({

        "Itemset":
        ", ".join(
            sorted(itemset)
        ),

        "Support":
        round(support, 4),

        "Count":
        count
    })


result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    by="Support",
    ascending=False
)

print("Top Frequent Itemsets:")

display(
    result_df.head(20)
)

Top Frequent Itemsets:


,Itemset,Support,Count
11,mineral water,426.2024,3196944
17,eggs,143.8582,1079080
27,spaghetti,84.4021,633100
23,french fries,77.3150,579940
31,chocolate,35.0855,263176
9,green tea,21.1869,158923
28,cookies,12.7035,95289
19,milk,9.9627,74730
51,escalope,4.6552,34919
50,ground beef,3.6439,27333
